In [1]:
# # !pip install fuzzywuzzy
# !/opt/conda/envs/rapids/bin/python -m pip install -U fuzzywuzzy
# !/opt/conda/envs/rapids/bin/python -m pip install -U --user openai
# !/opt/conda/envs/rapids/bin/python -m pip install -U --user vertexai

In [2]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')



import openai
import os
import json

def load_api_keys(root_dir):
    import json
    with open(f"{root_dir}/API_KEYS2.json", "r") as file:
        api_keys = json.load(file)
    os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
    os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
    # os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
    os.environ['LLMS_CACHE_PATH'] = "emotion_qwen-1"
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.path.join(root_dir, api_keys['GOOGLE_APPLICATION_CREDENTIALS'])
    os.environ['CACHE_DIR'] = os.path.join(root_dir, 'cache_dir3')
    return api_keys

load_api_keys(ROOT_DIR);

# Emotion

In [3]:
import importlib
import sys; sys.path.append("../src")
import emotion
importlib.reload(emotion)
from emotion import EmotionExample, get_llm_generated_answer, isolate_individual_features
from emotion import distill_relevant_features, calculate_expert_alignment_score
from emotion import load_emotion_data, run_pipeline, group_claims_by_category, make_alignment_matrix, categories_list #aggregate_alignment_scores
from llms import load_model
# from cholec import get_llm_generated_answer
# from cholec import CholecExample, CholecDataset, load_model, items_to_examples
# from cholec import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores

/opt/conda/envs/rapids/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
emotion_data =  load_emotion_data().sample(100, random_state=11)
emotion_data = emotion_data.reset_index(drop=True)

emotion_labels = {
    0: "admiration",
    1: "amusement",
    2: "anger",
    3: "annoyance",
    4: "approval",
    5: "caring",
    6: "confusion",
    7: "curiosity",
    8: "desire",
    9: "disappointment",
    10: "disapproval",
    11: "disgust",
    12: "embarrassment",
    13: "excitement",
    14: "fear",
    15: "gratitude",
    16: "grief",
    17: "joy",
    18: "love",
    19: "nervousness",
    20: "optimism",
    21: "pride",
    22: "realization",
    23: "relief",
    24: "remorse",
    25: "sadness",
    26: "surprise",
    27: "neutral"
}

emotion_data

,text,labels,id
0,What I know is you seem perfectly comfortable ...,[2],ed5rwg1
1,You can assume he means in the 5 man at least ...,[22],eemekvv
2,"Everyday, always, I am a hero inside my head",[21],edc9a8b
3,"It's been deleted, we can't even see what it w...",[25],eezjsmz
4,"aha American Sniper, movie genuinely moved me.",[0],ef1ff2k
...,...,...,...
95,"Lol, I don't know the game that well",[1],edje46l
96,The Tommy Gun as a crate weapon! Just the wors...,[11],edt038n
97,Wish they would’ve cancelled work when it was ...,[8],ef53auo
98,"""oh that wasn't me, my baby brother must have ...",[12],eemqadp


In [5]:
from tqdm.auto import tqdm
import json

In [6]:
# model = 'gpt-4o'
models = [
    "gpt-5.2-pro-2025-12-11",
    "gpt-5-mini-2025-08-07",
    "claude-opus-4-5-20251101",
    # "claude-haiku-4-5-20251001",
    # "gemini-2.5-pro",
    # "gemini-2.5-flash"
]

eval_model = load_model("Qwen/Qwen2.5-VL-7B-Instruct")
eval_model_name = 'qwen2.5-vl'


INFO 01-29 03:04:50 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 01-29 03:04:50 [__init__.py:239] Automatically detected platform cuda.
Loading Qwen-VL with vLLM: Qwen/Qwen2.5-VL-7B-Instruct
INFO 01-29 03:04:59 [config.py:717] This model supports multiple tasks: {'generate', 'score', 'classify', 'reward', 'embed'}. Defaulting to 'generate'.
INFO 01-29 03:04:59 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 01-29 03:05:00 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen2.5-VL-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-VL-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=F

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


WARNING 01-29 03:05:05 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
INFO 01-29 03:05:05 [gpu_model_runner.py:1329] Starting to load model Qwen/Qwen2.5-VL-7B-Instruct...
WARNING 01-29 03:05:05 [vision.py:93] Current `vllm-flash-attn` has a bug inside vision module, so we use xformers backend instead. You can run `pip install flash-attn` to use flash-attention backend.
INFO 01-29 03:05:05 [config.py:3614] cudagraph sizes specified by model runner [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 264, 272, 280, 288, 296, 304, 312, 320, 328, 336, 344, 352, 360, 368, 376, 384, 392, 400, 408, 416, 424, 432, 440, 448, 456, 464, 472, 480, 488, 496, 504, 512] is overridden by config [512, 384, 256, 128, 4, 2, 1, 392, 264, 136, 8, 400, 272, 144, 16, 408

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 01-29 03:05:09 [loader.py:458] Loading weights took 2.89 seconds
INFO 01-29 03:05:09 [gpu_model_runner.py:1347] Model loading took 15.6271 GiB and 3.561776 seconds
INFO 01-29 03:05:11 [gpu_model_runner.py:1620] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
INFO 01-29 03:05:19 [backends.py:420] Using cache directory: /home/runai-home/.cache/vllm/torch_compile_cache/6b4a7092f4/rank_0_0 for vLLM's torch.compile
INFO 01-29 03:05:19 [backends.py:430] Dynamo bytecode transform time: 5.34 s
INFO 01-29 03:05:23 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 3.299 s
INFO 01-29 03:05:28 [monitor.py:33] torch.compile takes 5.34 s in total
INFO 01-29 03:05:29 [kv_cache_utils.py:634] GPU KV cache size: 917,744 tokens
INFO 01-29 03:05:29 [kv_cache_utils.py:637] Maximum concurrency for 8,192 tokens per request: 112.03x
INFO 01-29 03:05:53 [gpu_model_runner.py:1686] Graph cap

In [7]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [8]:
1

1

In [9]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
import json
import copy
from tqdm.auto import tqdm

# for model in models:
#     print(f"=== Using model {model} ===")
#     for method in methods:
        # print(f"=== Using method {method} ===")

model = models[0]
method = methods[0]
    
load_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}.json')
save_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}_{eval_model_name}.json')

with open(load_path) as input_file:
    results = json.load(input_file)
    
results[0].keys()

dict_keys(['text', 'ground_truth', 'llm_label', 'llm_explanation', 'accuracy', 'claims', 'relevant_claims', 'claims_by_category', 'category_alignment_scores', 'alignment_matrix', 'final_alignment_score'])

In [11]:
emotion_data.iloc[0].to_dict()

{'text': 'What I know is you seem perfectly comfortable being a hypocrite but utterly outraged at the idea of admitting to it.',
 'labels': [2],
 'id': 'ed5rwg1'}

In [ ]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}_{eval_model_name}.json')
        

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            row = emotion_data.iloc[di].to_dict()
            
            # image = test_dataset[id2idx_mapping[result['id']]]['image']
            text = row['text']
            
            example = EmotionExample(
                text=row['text'],
                ground_truth=emotion_labels[row['labels'][0]],
                llm_label=result['llm_label'],
                llm_explanation=result['llm_explanation']
            )
            
            
            # isolate individual features
            claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if claims is None:
                continue
            example.claims = [claim.strip() for claim in claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example,
                model=eval_model
            )
            example.relevant_claims = relevant_claims

            print("----- Grouping claims by category -----")
            # for example in tqdm(examples):
            # print(example.text)
            claims_by_category = group_claims_by_category(example.relevant_claims, model=eval_model)
            example.claims_by_category = claims_by_category

            print("----- Calculating expert alignment scores -----")
            # for example in tqdm(examples):
            category_alignment_scores = calculate_expert_alignment_score(example.claims_by_category, model=eval_model)
            example.category_alignment_scores = category_alignment_scores
            example.alignment_matrix = make_alignment_matrix(categories_list, example.claims, example.claims_by_category, example.category_alignment_scores)
            final_aligned_score = example.alignment_matrix.max(axis=-1).mean()
            example.final_aligned_score = final_aligned_score
            
            
            # save
            import torch
            import numpy as np

            save_dict = {}

            for k, v in example.__dict__.items():
                if isinstance(v, torch.Tensor):
                    save_dict[k] = v.detach().cpu().numpy().tolist()
                elif isinstance(v, np.ndarray):
                    save_dict[k] = v.tolist()
                else:
                    save_dict[k] = v

            # with open(save_path, 'wt') as output_file:
            #     json.dump(save_dict, output_file)

            new_results.append(save_dict)
            

        with open(save_path, 'wt') as output_file:
            json.dump(new_results, output_file, indent=4)

=== Using model gpt-5.2-pro-2025-12-11 ===
=== Using method vanilla ===


  0%|          | 0/100 [00:00<?, ?it/s]


100%|██████████| 8/8 [00:00<00:00, 159.04it/s]

----- Grouping claims by category -----
----- Calculating expert alignment scores -----




100%|██████████| 10/10 [00:00<00:00, 160.38it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

# Cholec

# Emotion